In [1]:
import torch
import numpy as np
import pandas as pd
import re
import os
import matplotlib.pyplot as plt
import torchvision
import tqdm

import datasets
import training_loop
import losses
import models

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(device)

cuda


In [2]:
ef_reports = pd.read_csv("../labels/EF_reports.csv")
ef_reports = ef_reports.drop(columns=['MRN','SURGERY_DATE','JOINED_IMPRESSION','EF_post'])
ef_reports = ef_reports.dropna()

In [3]:
cycle1000 = pd.read_csv("../data/feedback_labeling/cycle_1000/predictions_all_cycle_1000.csv")
cycle2000 = pd.read_csv("../data/feedback_labeling/cycle_2000/predictions_all_cycle_2000.csv")
# cycle3000 = pd.read_csv("../data/feedback_labeling/cycle_3000/predictions_all_cycle_3000.csv")
cycle1000 = cycle1000.loc[cycle1000['prediction']==1]
cycle2000 = cycle2000.loc[cycle2000['prediction']==1]
# cycle3000 = cycle3000.loc[cycle3000['prediction']==1]

# cycle4000 = pd.read_csv("../data/feedback_labeling/cycle_4000/predictions_all_cycle_4000.csv")
# cycle4000 = cycle4000.loc[cycle4000['prediction']==1]

# train_cycles = pd.concat((cycle1000, cycle2000, ))
# test_cycles = cycle3000

train_cycles = cycle1000
test_cycles = cycle2000

In [4]:
# ef_reports = ef_reports.reset_index()
train_cycles = train_cycles.reset_index()
test_cycles = test_cycles.reset_index()

In [5]:
print(len(train_cycles))
removals = []
for i in range(len(train_cycles['clip_path'])):
    if int(os.path.basename(os.path.split(train_cycles['clip_path'].iloc[i])[0])) not in list(ef_reports['ACC_NUM']):
        removals.append(i)
# for r in removals[::-1]:
#     train_cycles = train_cycles.drop(r)
train_cycles = train_cycles.drop(removals)
print(len(train_cycles))

print(len(test_cycles))
removals = []
for i in range(len(test_cycles['clip_path'])):
    if int(os.path.basename(os.path.split(test_cycles['clip_path'].iloc[i])[0])) not in list(ef_reports['ACC_NUM']):
        removals.append(i)
# for r in removals[::-1]:
#     test_cycles = test_cycles.drop(r)
test_cycles = test_cycles.drop(removals)
print(len(test_cycles))

train_cycles = train_cycles.reset_index()
test_cycles = test_cycles.reset_index()

4593
4431
5408
5243


In [6]:
train_clips = train_cycles['clip_path']
test_clips = test_cycles['clip_path']

In [7]:
# train_transforms = datasets.TransformsComposed('train', size=(64,64))
# train_transforms = datasets.TransformsComposed('train', size=(128,128))
train_transforms = datasets.TransformsComposed('train', size=(224,224))
valtest_transforms = datasets.TransformsComposed('valtest', size=(224,224))

In [8]:
train_dataset = datasets.VideoDataset(clip_paths=train_clips, labels_df=ef_reports, transform=train_transforms, duration=16, skip=1)
test_dataset = datasets.VideoDataset(clip_paths=test_clips, labels_df=ef_reports, transform=valtest_transforms, duration=16, skip=1)